# Lab 07 — Root Cause Analysis with LLM

預警（Lab 06）回答「未來可能何時出事」。RCA 接著回答 **「為什麼會出事」**：哪個是根因、哪些只是下游症狀、證據是什麼。

本 lab 對應理論單元 08，實作四件事：

1. 計算變數間 **correlation** 與 **lagged correlation（lead-lag）**。
2. 結合時序、拓樸與領域知識，建立 **root cause ranking** 與 **evidence chain**。
3. 建立 **可解釋模型（RandomForest + SHAP）** 與 **LLM 輔助診斷**。
4. 產生 **事件摘要、根因候選與診斷說明**，並用 hit@k / MRR 評估。

RCA 不是讓 LLM 猜答案，而是把 metric、時序、拓樸與變更紀錄整理成 **可驗證的假說**。最終仍需要人類確認。

In [ ]:
from pathlib import Path
from IPython.display import SVG, display

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for base in (start, *start.parents):
        if (base / "environment.yml").exists():
            return base
    raise FileNotFoundError("Could not find project root containing environment.yml")

svg_candidates = [
    Path("diagrams/lab07_pipeline_position.svg"),
    find_project_root() / "labs/self-study" / "diagrams/lab07_pipeline_position.svg",
]
for svg_path in svg_candidates:
    if svg_path.exists():
        display(SVG(filename=str(svg_path)))
        break
else:
    raise FileNotFoundError("Could not find diagram: diagrams/lab07_pipeline_position.svg")

## Lab 07 概覽

### 學習目標

1. 區分 **symptom、proximate cause、contributing factor、root cause**，理解 detection / localization / diagnosis / remediation。
2. 用 **Pearson / Spearman correlation** 與 **cross-correlation（lead-lag）** 分析變數關係，並理解 correlation ≠ causation。
3. 結合 **anomaly onset、拓樸依賴、領域知識** 建立 root cause ranking 與 evidence chain。
4. 用 **RandomForest + SHAP** 建立可解釋模型，並理解 feature importance ≠ root cause。
5. 設計 **LLM 輔助診斷**（證據摘要 → 候選 → 說明），用工具與資料約束，避免幻覺。
6. 用 **hit@k、MRR、diagnosis coverage** 評估 RCA 品質。

### 前置條件

本 lab **自給自足**：直接讀取 `data/synthetic/synthetic_rrd_metrics.csv` + 事件目錄，在 notebook 內計算特徵，輸出寫到 `outputs/workshop/`，不需要先跑其他 lab。額外套件 `scipy`、`scikit-learn`、`networkx`、`shap`（見 `environments/environment.week6.yml`，additive）；`anthropic` 為選用（接真實 LLM 時才需要）。

## 設計主線：RCA 是證據組織，不是讓 LLM 猜答案

告警只說「哪裡不正常」。RCA 要把 metric、時序、拓樸、歷史事件與變更紀錄整理成可驗證的假說。

設計 RCA 架構時請問三個問題：

1. **context 是否足夠但不過量**：LLM 需要 **摘要後的 evidence**，不是整段 raw time series。
2. **假說是否可驗證**：每個 root-cause hypothesis 都應該對應到下一個檢查動作（查 switch log、SFP power、STP 狀態、最近部署）。
3. **輸出是否可被系統接住**：root cause ranking、confidence、evidence chain、recommended action 都要能進 ticket、webhook 或 Grafana。

設計原則：LLM 是把證據整理成候選解釋的助手，最後仍需要 **human approval gate**。

In [ ]:
from pathlib import Path
import json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

def show_fig(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "environment.yml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "outputs" / "workshop"     # workshop track output dir
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# optional dependencies (graceful)
try:
    import networkx as nx
    HAS_NX = True
except Exception:
    HAS_NX = False
try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report
    HAS_SKLEARN = True
except Exception:
    HAS_SKLEARN = False
try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

print(f"Project root: {PROJECT_ROOT}")
print(f"networkx={HAS_NX}  sklearn={HAS_SKLEARN}  shap={HAS_SHAP}")

In [ ]:
def build_features(df):
    """Derive the semantic features used in this lab from the raw RRD counters (cf. Lab 01)."""
    df = df.sort_values(["port_id", "timestamp"]).reset_index(drop=True)
    sdiv = lambda a, b: np.where(b > 0, a / b, 0.0)
    df["traffic_total"]   = df["INOCTETS"] + df["OUTOCTETS"]
    df["ucast_total"]     = df["INUCASTPKTS"] + df["OUTUCASTPKTS"]
    df["broadcast_total"] = df["INBROADCASTPKTS"] + df["OUTBROADCASTPKTS"]
    df["multicast_total"] = df["INMULTICASTPKTS"] + df["OUTMULTICASTPKTS"]
    df["errors_total"]    = df["INERRORS"] + df["OUTERRORS"]
    df["discards_total"]  = df["INDISCARDS"] + df["OUTDISCARDS"]
    df["error_rate"]      = sdiv(df["errors_total"], df["ucast_total"])
    df["discard_rate"]    = sdiv(df["discards_total"], df["ucast_total"])
    df["unknown_total"]   = df["INUNKNOWNPROTOS"]
    df["avg_packet_size"] = sdiv(df["traffic_total"], df["ucast_total"])
    return df

raw = pd.read_csv(DATA_SYNTHETIC / "synthetic_rrd_metrics.csv", parse_dates=["timestamp"])
features = build_features(raw)
events = pd.read_csv(DATA_SYNTHETIC / "synthetic_event_catalog.csv", parse_dates=["start_time", "end_time"])

# metrics that carry root-cause signature (see Lab 01 feature engineering)
METRICS = ["traffic_total", "error_rate", "discard_rate", "broadcast_total",
           "multicast_total", "unknown_total", "ucast_total", "avg_packet_size"]
ALL_PORTS = sorted(features.port_id.unique())
features = features.sort_values(["port_id", "timestamp"]).reset_index(drop=True)
feat_by_port = {p: g.set_index("timestamp") for p, g in features.groupby("port_id")}

# In production, RCA runs on the Lab 05 de-noised incident set. Here we use the synthetic
# event catalog's 10 events as the ground-truth incident set (the injected root cause is known),
# which lets us evaluate ranking with hit@k / MRR.
print(f"{len(events)} ground-truth incidents | {len(ALL_PORTS)} ports | "
      f"event_label is the simulator's per-row injected label (independent RCA ground truth)")
display(events[["event_id", "event_type", "port_id", "start_time", "end_time", "description"]])

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明，請先不要執行 cell。

---

## 📖 概念：症狀 vs 根因 — RCA Ladder

告警系統（Lab 02-05）回答「某個 metric 目前超出正常範圍」。這是 **症狀（symptom）**，不是 **根因（root cause）**。

| 名詞 | 意義 |
|---|---|
| Symptom | 可觀察的異常表現（API timeout、discard 上升） |
| Proximate cause | 直接造成症狀的近端原因（connection pool 耗盡） |
| Contributing factor | 放大事件機率 / 嚴重度的因素（容量餘裕不足、retry 過度積極） |
| **Root cause** | 移除 / 修正後能阻止事件或顯著降低重演的原因 |

### 從訊號到行動

```
Anomaly signal → Alert/Incident → Localization（哪裡） → Diagnosis（為什麼）
              → Evidence chain（哪些證據） → Decision / Remediation
```

AIOps 自動 RCA 主要做 **localization + diagnosis**。兩個常見陷阱：

- **最異常的指標不一定是根因**：下游症狀常比上游根因更劇烈（DB 小延遲 → 數十個服務大量 timeout）。
- **最早發生的異常也不一定是根因**：採樣延遲、時鐘偏差、資料缺漏都會干擾時間順序。

一個可信的根因候選通常 **同時** 具備：時間上較早、位於合理上游、能解釋多個症狀、與變更相符、移除後症狀應降低。

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 1 — 事件指標矩陣：每個 incident 每個 port 每個 metric 的異常程度

對每個 incident，比較事件窗口與事件前 baseline，算出每個 (port, metric) 的 robust z-score。z 越大代表該 port 在該 metric 上越異常。這是後續所有排名的證據基礎。

In [ ]:
def window_zscores(t0, t1, pre="2h"):
    """For one incident window, robust z = (event_mean - pre_mean)/pre_std per (port, metric)."""
    pre_td = pd.Timedelta(pre)
    rows = []
    for p in ALL_PORTS:
        g = feat_by_port[p]
        pre_w = g.loc[t0 - pre_td: t0 - pd.Timedelta("5min")]
        dur = g.loc[t0: t1]
        if pre_w.empty or dur.empty:
            continue
        rec = {"port_id": p}
        for m in METRICS:
            mu, sd = pre_w[m].mean(), pre_w[m].std()
            rec[m] = float((dur[m].mean() - mu) / sd) if sd and sd > 1e-9 else 0.0
        rows.append(rec)
    return pd.DataFrame(rows).set_index("port_id")

# show the metric z-score matrix for one cross-port event (broadcast storm = event G)
ev_G = events[events.event_id == "G"].iloc[0]
zmat = window_zscores(ev_G.start_time, ev_G.end_time)
print(f"Incident {ev_G.event_id} ({ev_G.event_type}, ports={ev_G.port_id})  window {ev_G.start_time} → {ev_G.end_time}")
display(zmat.round(1).style.background_gradient(cmap="Reds", axis=None))

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明，請先不要執行 cell。

---

## 📖 概念：Correlation 與 Causation

correlation 描述兩變數共同變化的程度，但高相關可能來自：$X$ 造成 $Y$、$Y$ 造成 $X$、共同原因 $Z$、共同季節性 / 趨勢、或純粹巧合。

$$r_{XY}=\frac{\sum_t(x_t-\bar{x})(y_t-\bar{y})}{\sqrt{\sum_t(x_t-\bar{x})^2}\sqrt{\sum_t(y_t-\bar{y})^2}}$$

- **Pearson**：線性關係，對離群值敏感。
- **Spearman**：對排名計算，適合單調但非線性，對極端值較穩健。
- **Mutual Information**：可捕捉非線性依賴，但無方向、估計需足夠資料。

因果關心的是 **介入**：$P(Y\mid do(X=x))$，主動改變 $X$ 時 $Y$ 是否改變。**相關性適合篩選候選與分群，不適合單獨宣稱根因。** 兩個都有上升趨勢的序列，即使無關也會得到高相關（虛假相關），所以分析前要先處理趨勢、季節性與自相關。

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 2 — 跨 port correlation（Pearson + Spearman）

對某個事件窗口，計算各 port 在 signature metric 上的 correlation 矩陣。廣播風暴這類 L2 事件會在多個 port 同步出現，correlation 會很高：這是「共同原因 / fan-out」的弱證據。

In [ ]:
def cross_port_corr(metric, t0, t1, method="pearson"):
    wide = pd.DataFrame({p: feat_by_port[p].loc[t0:t1][metric] for p in ALL_PORTS}).dropna(how="all")
    return wide.corr(method=method)

corr_p = cross_port_corr("broadcast_total", ev_G.start_time, ev_G.end_time, "pearson")
corr_s = cross_port_corr("broadcast_total", ev_G.start_time, ev_G.end_time, "spearman")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, c, name in zip(axes, [corr_p, corr_s], ["Pearson", "Spearman"]):
    im = ax.imshow(c.values, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(c))); ax.set_xticklabels(c.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(c))); ax.set_yticklabels(c.index, fontsize=8)
    ax.set_title(f"{name} corr — broadcast_total during {ev_G.event_type}")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); show_fig(fig)
print("高相關只是「一起變化」的證據，不代表其中一個 port 是另一個的原因。")

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明，請先不要執行 cell。

---

## 📖 概念：時序關係與 Lead-Lag

若 $X$ 的異常通常先出現、之後 $Y$ 才異常，可以說 $X$ **lead** $Y$。交叉相關比較不同時間位移下的關係：

$$CCF_{XY}(k)=Corr(X_t,\,Y_{t+k})$$

在正 lag $k$ 關係最高，表示 $X$ 領先 $Y$ 約 $k$ 個時間單位。**anomaly onset**（首次偏離 baseline 的時間）也是重要的方向線索。

限制：cross-correlation 會被共同季節性、自相關、不同採樣頻率、時鐘不同步影響。若兩序列都有強自相關，先移除各自可預測結構（**pre-whitening**）再比較殘差，可降低虛假 lead-lag。

**Granger causality**：若加入 $X$ 的過去值能顯著改善對 $Y$ 的預測，稱 $X$ Granger-causes $Y$。它回答的是「$X$ 的歷史是否對預測 $Y$ 有額外價值」，提供方向線索，但 **不等於真正因果**。

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 3 — Lagged correlation（lead-lag）與 anomaly onset

對一個 incident，計算 (a) 各 port 的 **anomaly onset**（signature metric 首次超過 baseline+3σ 的時間），以及 (b) 兩個 port 之間的 **cross-correlation lead-lag**。最早 onset 且能領先他人的 port，是傳播圖上游的候選。

> 本 lab 為求精簡，直接在事件窗口內用（標準化後的）原始序列計算 correlation / cross-correlation；production 應先 **detrend / pre-whitening**（移除趨勢、季節性、自相關）再比較殘差，否則容易得到虛假 lead-lag（見上方概念說明）。

In [ ]:
def anomaly_onset(port, metric, t0, t1, pre="2h", k=3):
    g = feat_by_port[port]
    pre_w = g.loc[t0 - pd.Timedelta(pre): t0 - pd.Timedelta("5min")][metric]
    mu, sd = pre_w.mean(), max(pre_w.std(), 1e-9)
    dur = g.loc[t0:t1][metric]
    crossed = dur[(dur - mu) / sd > k]
    return crossed.index[0] if len(crossed) else None

def lead_lag(x, y, max_lag=6):
    """Return lag (in 5-min steps) at which x best leads y; positive => x leads y."""
    x = (x - x.mean()) / (x.std() + 1e-9)
    y = (y - y.mean()) / (y.std() + 1e-9)
    n = min(len(x), len(y)); x, y = x.values[:n], y.values[:n]
    best_lag, best = 0, -np.inf
    for lag in range(-max_lag, max_lag + 1):
        if lag >= 0:
            a, b = x[: n - lag], y[lag:]
        else:
            a, b = x[-lag:], y[: n + lag]
        if len(a) > 3:
            c = np.corrcoef(a, b)[0, 1]
            if np.isfinite(c) and c > best:
                best, best_lag = c, lag
    return best_lag, best

# onset table for the queue-congestion incident (event D, single port)
ev_D = events[events.event_id == "D"].iloc[0]
sig = {"queue_congestion": "discard_rate", "link_quality_issue": "error_rate",
       "broadcast_storm": "broadcast_total", "multicast_flooding": "multicast_total",
       "large_file_backup": "traffic_total", "business_traffic_growth": "traffic_total",
       "small_packet_scan": "ucast_total", "abnormal_device_sender": "traffic_total",
       "unknown_protocol_scan": "unknown_total", "load_sensitive_link_issue": "error_rate"}
metric_D = sig[ev_D.event_type]
onsets = []
for p in ALL_PORTS:
    o = anomaly_onset(p, metric_D, ev_D.start_time, ev_D.end_time)
    onsets.append({"port_id": p, "metric": metric_D, "onset": o})
onset_df = pd.DataFrame(onsets).sort_values("onset", na_position="last")
print(f"Incident {ev_D.event_id} ({ev_D.event_type}, true port={ev_D.port_id}) — anomaly onset by port:")
display(onset_df)

# lead-lag between the true-cause port and another port during a cross-port event (G)
xb = feat_by_port[ALL_PORTS[0]].loc[ev_G.start_time:ev_G.end_time]["broadcast_total"]
yb = feat_by_port[ALL_PORTS[1]].loc[ev_G.start_time:ev_G.end_time]["broadcast_total"]
lag, c = lead_lag(xb, yb)
print(f"\nlead-lag {ALL_PORTS[0]} vs {ALL_PORTS[1]} (broadcast_total): best lag={lag} steps "
      f"({lag*5} min), corr={c:.2f}  -> {'former leads' if lag>0 else 'latter leads' if lag<0 else 'synchronous'}")

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明，請先不要執行 cell。

---

## 📖 概念：異常傳播、拓樸與知識圖譜

純統計方法不知道哪個服務呼叫哪個服務、哪些主機共用資料庫、哪個告警是上游。**拓樸（topology）** 把任意相關關係限制為系統上合理的傳播路徑。

- **共同資源 fan-out**：多個服務同時異常，可能指向共同依賴，而非多個獨立故障。
- **級聯故障 cascade**：小問題因 retry、queue、load balancing 放大。RCA 要區分 **初始觸發點** 與 **後續放大機制**。

傳播圖上的根因候選通常：位於異常子圖上游、時間較早、能到達多個症狀節點、與已知依賴方向一致、本身有異常或變更證據。

**知識圖譜（Knowledge Graph）** 進一步整合 CMDB、服務擁有者、依賴、告警規則、過往 incident、runbook、部署紀錄。本課程用一個小型 device→port 依賴圖示範拓樸如何約束候選。

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 4 — Root cause ranking 與 evidence chain

把證據組合成一個 **非飽和（non-saturating）** 的排名分數：

$$Score(c)=w_1 A(c)+w_2 T(c)+w_3 G(c)+w_4 C(c)+w_5 K(c)$$

其中 $A$=候選自身異常程度、$T$=時間領先、$G$=拓樸位置與 fan-out、$C$=與其他症狀的關聯、$K$=領域知識 / 變更。對每個 incident 排名候選 port，並組裝 evidence chain（onset 時間軸）。

In [ ]:
# small topology: device -> downstream device; ports belong to devices (L2 broadcast domain = device)
PORT_DEVICE = features.groupby("port_id").device_id.first().to_dict()
DEVICE_DOWNSTREAM = {"edge-fw-01": ["core-sw-01"], "core-sw-01": ["dist-sw-02"], "dist-sw-02": []}

def _closure(dev):                       # transitive downstream reach (fallback when no networkx)
    seen, stack = set(), list(DEVICE_DOWNSTREAM.get(dev, []))
    while stack:
        d = stack.pop()
        if d not in seen:
            seen.add(d); stack += DEVICE_DOWNSTREAM.get(d, [])
    return seen

if HAS_NX:
    G_topo = nx.DiGraph()
    G_topo.add_nodes_from(DEVICE_DOWNSTREAM)         # ensure leaf nodes exist for descendants()
    for dev, downs in DEVICE_DOWNSTREAM.items():
        for d in downs:
            G_topo.add_edge(dev, d)
    def downstream_devices(dev):
        return set(nx.descendants(G_topo, dev))      # graph genuinely drives the topology term
else:
    def downstream_devices(dev):
        return _closure(dev)

CROSS_PORT_METRICS = {"broadcast_total", "multicast_total"}   # L2 fan-out signatures

def rank_candidates(ev, w=(0.35, 0.25, 0.20, 0.10, 0.10)):
    t0, t1 = ev.start_time, ev.end_time
    metric = sig[ev.event_type]
    z = window_zscores(t0, t1)
    onset = {p: anomaly_onset(p, metric, t0, t1) for p in ALL_PORTS}
    onset_rank = sorted([p for p in ALL_PORTS if onset[p] is not None], key=lambda p: onset[p])
    anomalous = [p for p in ALL_PORTS if abs(z.loc[p, metric]) > 3]
    rows = []
    for p in ALL_PORTS:
        A = np.clip(abs(z.loc[p, metric]) / 8.0, 0, 1)                              # anomaly magnitude
        T = 1.0 - (onset_rank.index(p) / max(len(onset_rank), 1)) if p in onset_rank else 0.0  # earlier = higher
        # topology: fan-out for L2 metrics; otherwise upstream-ness
        if metric in CROSS_PORT_METRICS:
            same_dev = [q for q in anomalous if PORT_DEVICE[q] == PORT_DEVICE[p]]
            G = np.clip(len(same_dev) / max(len(anomalous), 1), 0, 1)
        else:
            downs = downstream_devices(PORT_DEVICE[p])       # full downstream reach via the topology graph
            reach = [q for q in anomalous if PORT_DEVICE[q] in downs]
            G = np.clip((0.5 + 0.5 * len(reach)) if p in anomalous else 0.0, 0, 1)
        C = np.clip(len(anomalous) / len(ALL_PORTS), 0, 1) if p in anomalous else 0.0  # co-occurrence
        K = 0.5 if p in anomalous else 0.0                                          # placeholder: change/history
        score = 100 * (w[0]*A + w[1]*T + w[2]*G + w[3]*C + w[4]*K)
        rows.append({"port_id": p, "device_id": PORT_DEVICE[p], "z_signature": round(z.loc[p, metric], 1),
                     "onset": onset[p], "A": round(A, 2), "T": round(T, 2), "G": round(G, 2),
                     "score": round(score, 1)})
    out = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    out["rank"] = out.index + 1
    return out, onset

def evidence_chain(ev, onset):
    items = sorted([(t, p) for p, t in onset.items() if t is not None])
    chain = [f"{t:%m-%d %H:%M}  {p} ({PORT_DEVICE[p]}) {sig[ev.event_type]} crosses baseline+3σ" for t, p in items]
    return chain

_rank_cache = {}
def rank_candidates_cached(ev):
    """Cache rank_candidates per incident — it does per-port window scans and is reused downstream."""
    if ev.event_id not in _rank_cache:
        _rank_cache[ev.event_id] = rank_candidates(ev)
    return _rank_cache[ev.event_id]

rank_D, onset_D = rank_candidates_cached(ev_D)
print(f"Incident {ev_D.event_id} ({ev_D.event_type}) — true root cause port = {ev_D.port_id}")
display(rank_D[["rank", "port_id", "device_id", "z_signature", "onset", "A", "T", "G", "score"]])
print("\nEvidence chain:")
for line in evidence_chain(ev_D, onset_D):
    print("  " + line)

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明，請先不要執行 cell。

---

## 📖 概念：可解釋 AI 與 RCA — feature importance ≠ root cause

XAI 可以說明模型為何把某筆資料判為某類異常：哪些 feature 對輸出影響最大。**SHAP** 把模型預測分解為各 feature 的貢獻。

但要小心：

> 「此 feature 推高了模型判斷」 ≠ 「此 feature 已被證明為根因」。

原因：feature 可能是下游症狀、可能代理真正原因、彼此高度相關，且模型學到的是 **預測關係** 而非因果關係。RCA 更需要 **local explanation**（這次 incident 受哪些證據影響），global explanation 則用來檢查模型是否依賴不合理訊號。

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 5 — 可解釋模型：RandomForest + SHAP

用 row-level 的 metric 特徵訓練一個分類器，把每個 5 分鐘樣本分類成 `event_label`（含 normal），再用 SHAP 看哪些 feature 驅動每一類。這展示「模型用哪些訊號做判斷」，但記得：**重要不等於根因**。

> ⚠ 注意：這裡用 **分層隨機切分** 只為示範 SHAP。相鄰 5 分鐘樣本高度自相關且來自同一事件，random split 會把同一事件的相鄰列同時放進 train / test，造成 **時間洩漏（temporal leakage）**，所以印出的 accuracy 是 **樂觀上界**，只用來看 global feature importance，不代表線上表現。嚴謹評估應按事件 / 時間順序切分。

In [ ]:
if HAS_SKLEARN:
    FEATS = ["traffic_total", "error_rate", "discard_rate", "broadcast_total",
             "multicast_total", "unknown_total", "ucast_total", "avg_packet_size"]
    df = features.dropna(subset=FEATS).copy()
    # balance: keep all event rows, downsample normal
    ev_rows = df[df.event_label != "normal"]
    nrm = df[df.event_label == "normal"].sample(n=min(4000, (df.event_label=="normal").sum()), random_state=0)
    data = pd.concat([ev_rows, nrm], ignore_index=True)
    X, y = data[FEATS], data["event_label"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
    rf = RandomForestClassifier(n_estimators=200, max_depth=12, class_weight="balanced",
                                random_state=0, n_jobs=-1)
    rf.fit(Xtr, ytr)
    acc = rf.score(Xte, yte)
    print(f"RandomForest test accuracy: {acc:.2f}  (multiclass over {y.nunique()} labels)")
    imp = pd.Series(rf.feature_importances_, index=FEATS).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(9, 4))
    imp.sort_values().plot(kind="barh", ax=ax, color="tab:purple")
    ax.set_title("RandomForest feature importance (global) — NOT proof of root cause")
    plt.tight_layout(); show_fig(fig)
else:
    print("scikit-learn 未安裝，略過可解釋模型。")

In [ ]:
if HAS_SKLEARN and HAS_SHAP:
    try:
        Xs = Xte.sample(n=min(300, len(Xte)), random_state=0)
        sv = shap.TreeExplainer(rf).shap_values(Xs)
        if isinstance(sv, np.ndarray) and sv.ndim == 3:      # newer shap: (n, features, classes)
            sv = [sv[:, :, i] for i in range(sv.shape[2])]
        shap.summary_plot(sv, Xs, plot_type="bar", class_names=list(rf.classes_), show=False, max_display=8)
        fig = plt.gcf(); fig.set_size_inches(9, 5)
        plt.title("SHAP feature contribution by class (global)"); plt.tight_layout(); show_fig(fig)
        print("SHAP 說明模型如何做判斷。要回到根因，仍需時序、拓樸與變更證據佐證。")
    except Exception as e:
        print(f"SHAP 略過（{type(e).__name__}）。feature importance 已在上一格顯示。")
elif HAS_SKLEARN:
    print("shap 未安裝，略過 SHAP（feature importance 已在上一格顯示）。")

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明，請先不要執行 cell。

---

## 📖 概念：LLM 輔助診斷

LLM 擅長整合文字與結構化情境：彙整告警 / log / 變更、產生 incident timeline、把技術訊號轉成易讀摘要、根據 **已檢索證據** 提出根因候選、整理支持與反對證據。

但 LLM **不是因果證明工具**，可能依語言模式產生看似合理但未被資料支持的解釋。因此：

- 不應僅靠 LLM 文字判定根因；每個結論都應連結原始證據。
- 重要數值由工具或查詢計算（correlation、lag、onset、拓樸），LLM 負責 **規劃查詢、整合證據、表達結論**。
- 不確定時保留多個候選；不可虛構不存在的 metric / log / 部署。

建議的可稽核輸出：incident summary、影響範圍、根因候選排名、每個候選的支持與反對證據、可能傳播路徑、需要補充的資料、建議下一步。本 lab 預設用 **離線確定性 diagnoser**（讓所有學員無需任何金鑰即可跑完），並保留一個 **可選的 Claude adapter**：設了 `ANTHROPIC_API_KEY` 才會呼叫真實模型。

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 6 — 為每個 incident 組裝 RCA context（摘要後的證據）

把前面算好的證據（onset、ranking、correlation、topology）整理成一個精簡 dict，作為 diagnoser 的輸入。重點是 **以最少欄位呈現最強訊號**，不是把 raw time series 全塞進去。

In [ ]:
def build_context(ev):
    rank, onset = rank_candidates_cached(ev)
    z = window_zscores(ev.start_time, ev.end_time)
    metric = sig[ev.event_type]
    anomalous = [p for p in ALL_PORTS if abs(z.loc[p, metric]) > 3]
    top = rank.iloc[0]
    corr = cross_port_corr(metric, ev.start_time, ev.end_time, "pearson")
    if len(corr) > 1:
        vals = corr.values[~np.eye(len(corr), dtype=bool)]          # off-diagonal; constant series -> NaN
        mean_offdiag = float(np.nanmean(vals)) if np.isfinite(vals).any() else 0.0
    else:
        mean_offdiag = 0.0
    return {
        "incident_id": ev.event_id,
        "time_window": f"{ev.start_time:%Y-%m-%d %H:%M} → {ev.end_time:%H:%M}",
        "signature_metric": metric,
        "affected_ports": anomalous,
        "cross_port_synchrony": round(mean_offdiag, 2),
        "top_candidate": {"port_id": top.port_id, "device_id": top.device_id,
                          "z_signature": top.z_signature, "score": top.score},
        "ranking": rank[["rank", "port_id", "z_signature", "onset", "score"]].head(3).to_dict("records"),
        "evidence_chain": evidence_chain(ev, onset)[:4],
        "recent_changes": "None recorded",   # placeholder: wire to change-management in production
    }

contexts = {ev.event_id: build_context(ev) for _, ev in events.iterrows()}
print("Sample RCA context (incident G, broadcast storm):")
print(json.dumps(contexts["G"], indent=2, ensure_ascii=False, default=str))

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 7 — LLM 診斷：離線 diagnoser + 可選 Claude adapter

`diagnose()` 預設用 **離線確定性邏輯** 把證據合成成 JSON（incident summary、ranked root-cause hypotheses、recommended actions、escalation）。若環境設了 `ANTHROPIC_API_KEY` 且 `USE_LLM=True`，則改呼叫 Claude，輸出相同 schema，方便比較。課程不綁定任何 provider。

In [ ]:
import os

ACTIONS = {
    "broadcast_storm": ["檢查 STP / spanning-tree 狀態", "找最近加入的 port 或 VLAN 變更", "必要時 shutdown 疑似 loop 的 port", "接取層啟用 PortFast + BPDU Guard"],
    "multicast_flooding": ["檢查 IGMP snooping / querier", "確認 IPTV 或 routing 行為", "限制 multicast 範圍"],
    "queue_congestion": ["檢查 QoS / queue / 頻寬", "找 top talkers 與排程備份", "評估擴容或限流"],
    "link_quality_issue": ["檢查線材 / SFP / NIC", "確認 duplex / speed mismatch", "查 CRC / error 計數"],
    "load_sensitive_link_issue": ["在高負載時段重現", "檢查線路與光衰", "評估鏈路備援"],
    "large_file_backup": ["確認備份排程", "評估離峰執行或限流"],
    "business_traffic_growth": ["確認業務成長是否預期", "評估容量規劃"],
    "small_packet_scan": ["檢查 port scan / 短連線 / DDoS 小封包", "比對來源 IP"],
    "abnormal_device_sender": ["定位異常發送設備", "檢查是否被入侵或設定錯誤"],
    "unknown_protocol_scan": ["檢查非標準協定來源", "比對掃描 / 新設備上線"],
}
TYPE_LABEL = {"broadcast_storm": "Broadcast storm / L2 loop", "multicast_flooding": "Multicast flooding",
    "queue_congestion": "Queue congestion / buffer pressure", "link_quality_issue": "Link quality issue",
    "load_sensitive_link_issue": "Load-sensitive link instability", "large_file_backup": "Large transfer / backup",
    "business_traffic_growth": "Business traffic growth", "small_packet_scan": "Small-packet scan / surge",
    "abnormal_device_sender": "Abnormal sender", "unknown_protocol_scan": "Unknown protocol / scan"}

SYSTEM_PROMPT = (
    "你是一位資深 AIOps 工程師，專長是網路設備的根因分析。"
    "你會收到一個事件的結構化證據（已由工具計算的 metric z-score、anomaly onset、cross-port 同步度、"
    "候選排名與證據鏈）。請只根據提供的證據推理，不要虛構不存在的 metric / log / 部署。"
    "輸出必須是嚴格 JSON，schema："
    '{"incident_summary": str, "problem_type": str, '
    '"root_cause_hypotheses": [{"rank": int, "candidate": str, "confidence": "High|Medium|Low", "reasoning": str}], '
    '"supporting_evidence": [str], "contrary_evidence": [str], '
    '"recommended_actions": [str], "escalation_needed": bool}. 不要輸出 JSON 以外的任何文字。'
)

def diagnose_offline(ctx, ev_type):
    top = ctx["top_candidate"]
    sync = ctx["cross_port_synchrony"]
    multi = len(ctx["affected_ports"]) > 1
    summary = (f"{ctx['time_window']} 期間，{top['port_id']}（{top['device_id']}）在 {ctx['signature_metric']} "
               f"出現 z={top['z_signature']} 的異常；共 {len(ctx['affected_ports'])} 個 port 受影響"
               f"（cross-port 同步度 {sync}）。")
    # confidence from EVIDENCE COMPLETENESS, not the raw ranking score
    # (handout §15.4: 信心應反映證據完整性，而非只把排名分數轉成百分比)
    signals = 0
    signals += 1 if pd.notna(ctx["ranking"][0].get("onset")) else 0   # has a clear anomaly onset
    signals += 1 if sync > 0.5 else 0                                  # cross-port corroboration
    signals += 1 if multi else 0                                       # fan-out evidence
    signals += 1 if abs(top["z_signature"]) >= 5 else 0               # strong signature metric
    base_conf = "High" if signals >= 3 else "Medium" if signals == 2 else "Low"
    hyps = []
    for i, r in enumerate(ctx["ranking"]):
        conf = base_conf if i == 0 else ("Medium" if base_conf == "High" else "Low")
        why = (f"{signals}/4 corroborating signals (onset / cross-port sync / fan-out / strong z); "
               f"score={r['score']}; {'多 port 同步指向共同 L2 來源' if multi else '單 port 局部異常'}")
        hyps.append({"rank": r["rank"], "candidate": f"{r['port_id']} — {TYPE_LABEL.get(ev_type, ev_type)}",
                     "confidence": conf, "reasoning": why})
    support = [f"signature metric {ctx['signature_metric']} z={top['z_signature']}",
               f"最早 onset：{ctx['evidence_chain'][0] if ctx['evidence_chain'] else 'n/a'}"]
    if multi:
        support.append(f"{len(ctx['affected_ports'])} 個 port 同步異常，同步度 {sync}")
    contrary = [] if multi else ["僅單一 port 異常，缺少跨 port 佐證"]
    return {"incident_summary": summary, "problem_type": TYPE_LABEL.get(ev_type, ev_type),
            "root_cause_hypotheses": hyps, "supporting_evidence": support, "contrary_evidence": contrary,
            "recommended_actions": ACTIONS.get(ev_type, ["檢視 metric 趨勢、受影響 port、最近變更"]),
            "escalation_needed": bool(multi or top["score"] >= 60)}

def diagnose_claude(ctx, model=None):
    import anthropic
    model = model or os.environ.get("ANTHROPIC_MODEL", "claude-opus-4-8")
    client = anthropic.Anthropic()
    prompt = "事件證據（JSON）：\n" + json.dumps(ctx, ensure_ascii=False, default=str) + "\n\n請輸出診斷 JSON。"
    msg = client.messages.create(model=model, max_tokens=1500, system=SYSTEM_PROMPT,
                                 messages=[{"role": "user", "content": prompt}])
    text = "".join(b.text for b in msg.content if b.type == "text").strip()
    if text.startswith("```"):
        text = text.split("```", 2)[1].lstrip("json").strip()
    return json.loads(text)

USE_LLM = bool(os.environ.get("ANTHROPIC_API_KEY")) and os.environ.get("USE_LLM", "0") == "1"

def diagnose(ev):
    ctx = contexts[ev.event_id]
    if USE_LLM:
        try:
            return diagnose_claude(ctx)
        except Exception as e:
            print(f"  [LLM fallback -> offline] {ev.event_id}: {e}")
    return diagnose_offline(ctx, ev.event_type)

print(f"LLM mode: {'Claude (' + os.environ.get('ANTHROPIC_MODEL','claude-opus-4-8') + ')' if USE_LLM else 'offline deterministic'}")
diag_G = diagnose(events[events.event_id == "G"].iloc[0])
print(json.dumps(diag_G, indent=2, ensure_ascii=False))

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 8 — 評估：hit@k、MRR、diagnosis coverage

用 **獨立的真實根因** 評估排名品質：單 port 事件取自事件目錄；MULTI 事件取自 **模擬器的 per-row `event_label`**（與排名用的 z-score 獨立，避免循環）。

- **Hit@k**：真實根因是否落在前 $k$ 個候選。
- **MRR** $=\frac{1}{N}\sum 1/rank_i$：真實根因排名越前越高。
- **Diagnosis coverage**：有多少 incident 能產生有效候選與證據。

> ⚠ 兩個前提：(1) 這是 **離線評估**，看得到完整事件資料；線上只看得到當下已到達的部分資料，會更難。(2) MULTI（broadcast / multicast）這類 L2 事件，整個廣播域都算根因，hit@1 自然偏高。完美的 1.0 在這份乾淨合成資料上是 **預期** 的，不代表線上也這麼容易。

In [ ]:
def truth_ports(ev):
    """INDEPENDENT ground truth: the ports the simulator actually injected this event into
    (features.event_label), NOT re-derived from the same z-scores that drive the ranking.
    This avoids the circularity that would otherwise make Hit@1 tautological for MULTI events."""
    if ev.port_id != "MULTI":
        return {ev.port_id}
    w = features[(features.timestamp >= ev.start_time) & (features.timestamp <= ev.end_time)
                 & (features.event_label == ev.event_type)]
    return set(w.port_id.unique())   # may be empty -> counted as no-coverage, not an automatic hit

eval_rows = []
for _, ev in events.iterrows():
    rank, _ = rank_candidates_cached(ev)
    truth = truth_ports(ev)
    ranks_of_truth = [int(r["rank"]) for _, r in rank.iterrows() if r["port_id"] in truth]
    best = min(ranks_of_truth) if ranks_of_truth else None
    diag = diagnose(ev)
    eval_rows.append({
        "incident": ev.event_id, "event_type": ev.event_type, "true_port": ev.port_id,
        "pred_top": rank.iloc[0].port_id, "rank_of_true": best,
        "hit@1": int(best == 1) if best else 0, "hit@3": int(best is not None and best <= 3),
        "rr": (1.0 / best) if best else 0.0,
        "n_hypotheses": len(diag["root_cause_hypotheses"]),
    })
ev_df = pd.DataFrame(eval_rows)
display(ev_df)
hit1, hit3, mrr = ev_df["hit@1"].mean(), ev_df["hit@3"].mean(), ev_df["rr"].mean()
coverage = (ev_df["n_hypotheses"] > 0).mean()
print(f"\nHit@1={hit1:.2f}  Hit@3={hit3:.2f}  MRR={mrr:.2f}  Diagnosis coverage={coverage:.0%}  (N={len(ev_df)})")

fig, ax = plt.subplots(figsize=(7, 4))
pd.Series({"Hit@1": hit1, "Hit@3": hit3, "MRR": mrr, "Coverage": coverage}).plot(kind="bar", ax=ax, color="tab:green")
ax.set_ylim(0, 1); ax.set_title("RCA ranking quality"); ax.set_ylabel("score")
plt.tight_layout(); show_fig(fig)

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 9 — 事件摘要、根因候選與診斷說明 + 輸出 Grafana 數值

對每個 incident 產生人類可讀的診斷報告（摘要 + 候選 + 證據 + 建議），並把 **數值欄位** 存成 `rca_results.csv`（drop-zone 相容：severity、root_cause_score、rank_of_true、hit@3、confidence、escalation）。**長文字說明留在 notebook，不進 Prometheus。**

In [ ]:
def render_report(ev):
    diag = diagnose(ev); ctx = contexts[ev.event_id]
    lines = [f"=== Incident {ev.event_id} — {diag['problem_type']} ===",
             f"Summary: {diag['incident_summary']}",
             "Root-cause candidates:"]
    for h in diag["root_cause_hypotheses"]:
        lines.append(f"  #{h['rank']} [{h['confidence']}] {h['candidate']}  ({h['reasoning']})")
    lines.append("Supporting: " + "; ".join(diag["supporting_evidence"]))
    if diag["contrary_evidence"]:
        lines.append("Contrary: " + "; ".join(diag["contrary_evidence"]))
    lines.append("Actions: " + "; ".join(diag["recommended_actions"][:3]))
    lines.append(f"Escalation: {diag['escalation_needed']}")
    return "\n".join(lines)

print(render_report(events[events.event_id == "D"].iloc[0]))
print()
print(render_report(events[events.event_id == "G"].iloc[0]))

# numeric RCA results for the drop zone (one row per incident; long text stays in the notebook)
rca_rows = []
for _, ev in events.iterrows():
    rank, _ = rank_candidates_cached(ev); diag = diagnose(ev); ctx = contexts[ev.event_id]
    top = rank.iloc[0]; er = ev_df[ev_df.incident == ev.event_id].iloc[0]
    conf = diag["root_cause_hypotheses"][0]["confidence"] if diag["root_cause_hypotheses"] else "Low"
    rca_rows.append({
        "timestamp": ev.start_time, "device_id": top.device_id, "port_id": top.port_id,
        "port_role": features[features.port_id == top.port_id].port_role.iloc[0],
        "event_label": ev.event_type, "ml_method": "rca",
        "root_cause_score": float(top.score),
        "rank_of_true_cause": float(er["rank_of_true"]) if er["rank_of_true"] else float("nan"),
        "hit_at_3": float(er["hit@3"]),
        "affected_port_count": float(len(ctx["affected_ports"])),
        "cross_port_synchrony": float(ctx["cross_port_synchrony"]),
        "confidence_score": {"High": 0.9, "Medium": 0.6, "Low": 0.3}[conf],
        "escalation_flag": float(diag["escalation_needed"]),
    })
rca_results = pd.DataFrame(rca_rows)
rca_results.to_csv(DATA_PROCESSED / "rca_results.csv", index=False)
print(f"\nsaved: {DATA_PROCESSED / 'rca_results.csv'}  ({len(rca_results)} incidents)")
display(rca_results[["timestamp", "port_id", "event_label", "root_cause_score", "rank_of_true_cause",
                     "hit_at_3", "confidence_score", "escalation_flag"]])

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明，請先不要執行 cell。

---

## 📖 概念：把 RCA 數值送進 Grafana（drop zone，additive）

RCA 的 **長文字 explanation 不適合放進 Prometheus**。原則：**數值結果進 Prometheus，文字脈絡放 notebook / ticket / report。**

跟 Lab 06 一樣走 drop zone：把 `rca_results.csv` 複製到 `outputs/prometheus-dropzone/current_results.csv`，`infra/python_results_exporter.py` 會把 `root_cause_score`、`confidence_score`、`escalation_flag`、`hit_at_3` 等數值曝露成 `aiops_python_result`，再由 Grafana 呈現。

**additive，不破壞既有設定**：原本的 dashboard / alerts.yml 保持不動；forecast 與 RCA 風險共用 **新的 dashboard** `infra/grafana/dashboards/forecast_rca_risk.json`（uid `aiops-forecast-rca`）。Grafana 也可用 **annotation** 標記 incident 時間點，讓值班人員把 RCA 結果疊在 metric 圖上。

關於 Grafana 的 LLM 整合（如把摘要顯示在 panel），可參考 Grafana ML LLM app：<https://grafana.com/docs/grafana-cloud/machine-learning/llm/llm-setup/>。

## Optional — 在 Grafana 看 RCA 數值（drop zone）

RCA 的長文字不進 Prometheus；只掛數值欄位。保持 `python infra/python_results_exporter.py` 執行，然後複製：

```bash
cp outputs/workshop/rca_results.csv outputs/prometheus-dropzone/current_results.csv
```

要曝露 RCA 數值欄位，啟動 exporter 時指定：

```bash
RESULTS_CSV_PATH=outputs/workshop/rca_results.csv \
RESULT_COLUMNS=root_cause_score,confidence_score,escalation_flag,hit_at_3,cross_port_synchrony \
python infra/python_results_exporter.py
```

建議 PromQL：`aiops_python_result{column="root_cause_score"}`、`aiops_python_result{column="escalation_flag"}`。匯入新 dashboard `infra/grafana/dashboards/forecast_rca_risk.json`（uid `aiops-forecast-rca`）即可；原 dashboard 不受影響。完整流程見 `labs/getting-started/05-prometheus-dropzone.md`。

---

## ⚖️ 方法比較：RCA 方法總表

| 方法 | 使用資訊 | 優點 | 主要限制 | 適合角色 |
|---|---|---|---|---|
| Correlation | 同步數值關係 | 快速、易計算 | 無方向、非因果 | 候選篩選 |
| Lagged correlation | 時間位移關係 | 提供 lead-lag 線索 | 受自相關與季節性影響 | 傳播分析 |
| Mutual information | 一般統計依賴 | 可捕捉非線性 | 無方向、估計較難 | 關聯探索 |
| Granger causality | 過去值的預測力 | 提供方向性線索 | 不等於真正因果 | 時序候選圖 |
| Rule-based | 專家規則 | 易解釋、精準 | 維護成本高 | 已知故障模式 |
| Topology-based | 系統依賴 | 符合結構、可追路徑 | 拓樸可能過期 | 上下游定位 |
| Causal graph / SCM | 因果假設 | 支援介入與反事實 | 假設強、識別困難 | 深度診斷 |
| Bayesian Network | 圖 + 條件機率 | 表達不確定性、整合先驗 | 條件機率難估、推論成本高 | 機率診斷 |
| XAI / SHAP | 模型解釋 | 說明模型判斷 | 重要性不等於根因 | 模型證據 |
| LLM | 文字與多源情境 | 摘要、整合、說明 | 幻覺與不可驗證推論 | 診斷助手 |
| **Hybrid（本 lab）** | 多源證據 | 較符合實務 | 系統整合複雜 | 生產級 RCA |

> 本 lab 實際計算的是 **Pearson / Spearman + lagged correlation + 拓樸排名**；Mutual Information、Granger causality、Bayesian Network、因果圖 / SCM 屬 **概念補充**，未在此計算（spec 只要求 correlation + lagged correlation）。

本 lab 是 **hybrid**：統計關聯篩選 + 時序先後排序 + 拓樸限制 + 領域知識，最後由 LLM 整合成可讀診斷。沒有單一方法能獨立宣稱根因。

## ⚠ 人類驗證點 #7 — RCA 診斷是假說，不是事實

排名第一、confidence=High，仍然是 **根據資料推論出的假說**。執行 runbook 前必須由人類確認。

| 系統輸出 | 人類需要驗證的事 |
|---|---|
| `top_candidate: port-X` | 實際登入交換器確認 `show spanning-tree` / SFP power |
| `recommended_actions: shutdown port` | 確認影響範圍，是否有 critical 服務在上面 |
| `escalation_needed: true` | 判斷業務衝擊等級，決定叫醒哪個 on-call |
| SHAP 顯示 `discard_rate` 最重要 | 這是 **模型重要性**，不是根因；要回到 onset / 拓樸佐證 |

### 常見失敗型態

1. **把高相關當根因**：共同季節性、流量與趨勢都會造成高相關。
2. **把最早 / 最嚴重異常當根因**：採樣延遲、時鐘偏差、下游放大都會誤導。
3. **忽略共同原因與系統變更**：兩個 port 同時壞，常是共同上游或剛部署。
4. **Feature importance 因果化**：模型重要特徵只是預測貢獻。
5. **LLM 幻覺**：所有關鍵主張都要連到資料來源；數值由工具算，不讓 LLM 自己編。
6. **離線評估 ≠ 線上**：離線可看到完整事件資料，線上只看得到當下已到達的資訊。

---
🔧 **探索練習** — 修改指定參數，觀察結果變化。

---

## 🔧 探索練習

1. 在 Step 4 的 `rank_candidates` 調整權重 `w=(0.35, 0.25, 0.20, 0.10, 0.10)`：把拓樸權重 $w_3$ 調大或調小，看 Hit@1 / MRR 怎麼變。哪些事件對拓樸權重最敏感？
2. 在 Step 3 把 `anomaly_onset` 的門檻 `k=3` 改成 `k=2` 或 `k=4`，觀察 onset 排序與 evidence chain 的變化。
3. 接真實 LLM：在環境設 `ANTHROPIC_API_KEY` 與 `USE_LLM=1`（可選 `ANTHROPIC_MODEL`），重跑 Step 7，比較 Claude 與離線 diagnoser 的 reasoning 與 confidence。

> 「如果模型診斷準確率 70%，但 rule-based 80%，你還需要模型嗎？在什麼條件下它是必要的？」

> 「broadcast_storm 是 MULTI 事件，為什麼用『同步度 + 同 device fan-out』比『最早 onset』更能定位？」

> 「feature importance 說 `discard_rate` 最重要。你會直接去修 discard 嗎？還是先看 onset 與拓樸？」

## 整合回顧：Lab 07 在 Pipeline 裡的位置

```
What you did in Lab 07                 Production equivalent
──────────────────────────            ─────────────────────────────────
window_zscores / correlation   →      evidence builder (query Prometheus over the incident window)
lead_lag / anomaly_onset       →      temporal propagation analysis
rank_candidates / evidence chain →    root-cause ranking service
RandomForest + SHAP            →      model evidence (importance, NOT proof)
diagnose() offline + Claude    →      provider adapter (offline fallback + LLM)
hit@k / MRR                    →      RCA quality dashboard
rca_results.csv                →      drop zone -> Prometheus -> new Grafana dashboard

⚠ Human verification point #7  →      NOC engineer confirms before running runbook
```

**Lab 07 輸出**：

- `rca_results.csv` — 每個 incident 的數值結果（root_cause_score、rank_of_true、hit@3、confidence、escalation；drop-zone 相容）。
- 人類可讀診斷報告（摘要 + 候選 + 證據 + 建議）：留在 notebook / ticket。

**整條智慧監控流程（Lab 00→07）**：Observe → Detect（SPC / ML / Forecasting）→ Correlate（降噪）→ **Diagnose（統計 + 拓樸 + LLM，本 lab）** → Decide → Act → Learn。預測說「未來可能何時出事」，RCA 說「為什麼」，最後仍由人類拍板。